# FITS File — All Headers Explorer
Run this notebook on your JWST `.fits` file to see every header from every extension.

In [ ]:
from astropy.io import fits
from pathlib import Path

original_path = Path(r"T:\JWST-Timing\MAST_2023-10-07T0548\JWST")

## Step 1 — Loop over all files and frames (fixed)
**Bug fixed:** `range(istart, iend+1)` → `range(istart, iend)` to avoid index out of bounds on the last frame.

In [10]:
for epoch in ["epoch1", "epoch2"]:
    for wave_type in ["lw", "sw"]:

        if epoch == "epoch1":
            sw_files = sorted(original_path.glob("jw01666001001*_nrcb1/*crfints.fits"))
            lw_files = sorted(original_path.glob("jw01666001001*_nrcblong/*crfints.fits"))
        else:
            sw_files = sorted(original_path.glob("jw01666003001*_nrcb1/*crfints.fits"))
            lw_files = sorted(original_path.glob("jw01666003001*_nrcblong/*crfints.fits"))

        fileList = lw_files if wave_type == "lw" else sw_files

        for filename in fileList:
            print(f"Processing {filename}...")

            with fits.open(filename) as f:
                data = f["SCI"]

                istart = 0
                iend = data.data.shape[0]  # total number of frames

                for i in range(istart, iend):
                    print(f"  File: {filename.name}   Frame: {i + 1} / {iend}")

## Step 2 — Show all headers from the first file found

In [7]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

# Grab the first available file to inspect headers
all_files = (
    list(original_path.glob("jw01666001001*_nrcb1/*crfints.fits")) +
    list(original_path.glob("jw01666001001*_nrcblong/*crfints.fits"))
)

if not all_files:
    print("No files found. Check your original_path.")
else:
    filepath = all_files[0]
    print(f"Inspecting: {filepath.name}\n")

    with fits.open(filepath) as f:
        print("Extensions in this file:")
        f.info()
        print()

        for i, hdu in enumerate(f):
            ext_name = hdu.name if hdu.name else f"Extension {i}"
            print(f"\n{'='*60}")
            print(f"  EXTENSION {i}: {ext_name}")
            print(f"{'='*60}")
            rows = [
                {
                    "Keyword": key,
                    "Value": str(value),
                    "Comment": hdu.header.comments[key]
                }
                for key, value in hdu.header.items()
                if key.strip()
            ]
            if rows:
                display(pd.DataFrame(rows))
            else:
                print("  (no headers)")

No files found. Check your original_path.


## Step 3 — Export all headers to CSV

In [8]:
if all_files:
    all_rows = []
    with fits.open(filepath) as f:
        for i, hdu in enumerate(f):
            ext_name = hdu.name if hdu.name else f"Extension {i}"
            for key, value in hdu.header.items():
                if key.strip():
                    all_rows.append({
                        "Extension": ext_name,
                        "Keyword": key,
                        "Value": str(value),
                        "Comment": hdu.header.comments[key]
                    })

    df_all = pd.DataFrame(all_rows)
    df_all.to_csv("fits_headers.csv", index=False)
    print(f"Saved {len(df_all)} header entries to fits_headers.csv")
    display(df_all)